In [2]:
# %% [markdown]
# # 📊 Posições com Cotação de Mercado
#
# Valor de mercado em tempo real, lucro/prejuízo e rentabilidade.

# %%
import sys
from pathlib import Path

# Adiciona a raiz do projeto ao path do Python
ROOT = Path.cwd()
# Se o notebook está em src/valuation/, sobe 2 níveis
if ROOT.name == "valuation":
    ROOT = ROOT.parent.parent
elif ROOT.name in ("src", "notebooks"):
    ROOT = ROOT.parent

sys.path.insert(0, "..")

# %%
from src.collectors.supabase_client import load_transactions
from src.portfolio.positions import calculate_positions, get_open_positions
from src.portfolio.market_data import enrich_with_market_data, portfolio_totals
from src.reports.accessible import market_summary


# %% [markdown]
# ## 1. Carregar transações e calcular posições

# %%
df = load_transactions()
posicoes = calculate_positions(df)
abertas = get_open_positions(posicoes)
print(f"✅ {len(abertas)} posições em aberto.")

# %% [markdown]
# ## 2. Buscar cotações e enriquecer

# %%
print("⏳ Buscando cotações no Yahoo Finance...")
enriched = enrich_with_market_data(abertas)
totals = portfolio_totals(enriched)
print("✅ Cotações carregadas!")

# %% [markdown]
# ## 3. Resumo completo com mercado

# %%
print(market_summary(enriched, totals))

# %% [markdown]
# ## 4. Tabela completa (DataFrame)

# %%
enriched[[
    "ticker", "nome", "categoria", "moeda",
    "qtde_saldo", "preco_medio_brl", "preco_atual_brl",
    "custo_total_brl", "valor_mercado_brl",
    "lucro_prejuizo_brl", "rentabilidade_pct",
]].sort_values("valor_mercado_brl", ascending=False)

# %% [markdown]
# ## 5. Maiores ganhos e perdas

# %%
print("=" * 60)
print("🟢 MAIORES GANHOS")
print("=" * 60)
gains = enriched[enriched["lucro_prejuizo_brl"] > 0].nlargest(5, "lucro_prejuizo_brl")
for _, r in gains.iterrows():
    rent = f"{r['rentabilidade_pct']:+.1f}%"
    print(f"  📈 {r['ticker']:12s} | L/P: {r['lucro_prejuizo_brl']:>12,.2f} BRL | Rent: {rent}")

print()
print("=" * 60)
print("🔴 MAIORES PERDAS")
print("=" * 60)
losses = enriched[enriched["lucro_prejuizo_brl"] < 0].nsmallest(5, "lucro_prejuizo_brl")
for _, r in losses.iterrows():
    rent = f"{r['rentabilidade_pct']:+.1f}%"
    print(f"  📉 {r['ticker']:12s} | L/P: {r['lucro_prejuizo_brl']:>12,.2f} BRL | Rent: {rent}")


✅ 75 posições em aberto.
⏳ Buscando cotações no Yahoo Finance...


Erro ao buscar USD/BRL: float() argument must be a string or a real number, not 'Series'. Usando fallback 5.20.
$IPCA+: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$ARB.SA: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$FET.SA: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$PREFIXADO: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$2029.SA: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$ETH.SA: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$BTC.SA: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$2065.SA: possibly delisted; no price data found  (period=2d) (Yah

✅ Cotações carregadas!
📊 CARTEIRA — VISÃO DE MERCADO

💰 Total investido:    R$ 181.115,40
🏦 Valor de mercado:   R$ 196.993,37
📈 Lucro/Prejuízo:     R$ 15.877,97 (+8,77%)
💵 Dólar (USD/BRL):    R$ 5.2000
📂 Posições abertas:   75

📁 POSIÇÕES COM COTAÇÃO ATUAL

── Ações (22 ativos | 58.9% | Mercado: R$ 116.068,46 | 📈 R$ 25.854,79) ──

  • ITSA4        | Qtde:       1138 | Atual: R$ 14,18 | Dia: +1,43% | Mercado: R$ 16.136,84 | 📈 R$ 3.208,84 (+24,82%) | Peso: 8.2%
  • SAPR11       | Qtde:        300 | Atual: R$ 45,80 | Dia: +2,85% | Mercado: R$ 13.740,00 | 📈 R$ 7.976,70 (+138,41%) | Peso: 7.0%
  • USIM5        | Qtde:       1600 | Atual: R$ 6,77 | Dia: +0,45% | Mercado: R$ 10.832,00 | 📈 R$ 251,53 (+2,38%) | Peso: 5.5%
  • CPLE3        | Qtde:        650 | Atual: R$ 15,83 | Dia: +2,59% | Mercado: R$ 10.289,50 | 📈 R$ 4.145,00 (+67,46%) | Peso: 5.2%
  • BBDC4        | Qtde:        520 | Atual: R$ 19,41 | Dia: +1,36% | Mercado: R$ 10.093,73 | 📈 R$ 3.078,33 (+43,88%) | Peso: 5.1%
  • TTEN3      